In [ ]:
from pathlib import Path
import sys

import pandas as pd

# Папка, где лежит last_experiment.py.
# Если ноутбук открыт не из этой папки, укажи путь вручную.
STUDIES_SRC = Path.cwd()
if not (STUDIES_SRC / "last_experiment.py").exists():
    STUDIES_SRC = Path(
        "/home/jovyan/sample_complexity_of_feature_selection_in_deep_tabular_models/src/net_complexity/studies"
    )

if str(STUDIES_SRC) not in sys.path:
    sys.path.insert(0, str(STUDIES_SRC))

from last_experiment import (
    collect_runs,
    load_history,
    summarize_one_run,
    plot_metric,
    plot_metrics,
    plot_gradient_norms,
    plot_acc_epoch,
)

# Путь до study/run
STUDY_DIR = Path(
    "../../../outputs/studies/PUT_STUDY_HERE/"
).resolve()

print("STUDY_DIR:", STUDY_DIR)
print("exists:", STUDY_DIR.exists())

# Поддерживаем оба случая:
# 1. STUDY_DIR — это study с несколькими run-директориями
# 2. STUDY_DIR — это один конкретный run с history.csv
if (STUDY_DIR / "history.csv").exists():
    history_df = load_history(STUDY_DIR)
    summary_df = pd.DataFrame([summarize_one_run(STUDY_DIR)])
else:
    summary_df, history_df = collect_runs(STUDY_DIR)

# В старом ноутбуке это называлось summary
summary = summary_df

print("history_df:", history_df.shape)
display(history_df.head())

print("summary_df:", summary_df.shape)
display(summary_df)


In [ ]:
# One call for a standard set of plots. Edit only this metric list.
METRICS = ["valid_accuracy", "valid_loss"]
plot_metrics(history_df, [metric for metric in METRICS if metric in history_df])

# For runs with gradient-norm logging:
# plot_gradient_norms(history_df, yscale="log")

In [ ]:
plot_acc_epoch(
    history_df,
    show_mean=False,
)

In [ ]:
plot_metric(
    history_df,
    "valid_active_blocks_expected",
    title="valid_active_blocks_expected / epoch",
)

In [ ]:
plot_metric(
    history_df,
    "valid_mean_gate_prob",
    title="valid_mean_gate_prob / epoch",
)

In [ ]:
plot_metric(
    history_df,
    "lambda_coef",
    title="lambda_coef / epoch",
    yscale="log",
)

## Entropy contribution checks

Новые графики ниже добавлены после старого набора. Одна ячейка строит один график.

In [ ]:
# Derived columns for entropy/lambda contribution plots.
import numpy as np

history_df = history_df.copy()
summary_df = summary_df.copy()

beta_col = next(
    (
        col for col in (
            "model.entropy_regularization_coef",
            "label_entropy_coef",
            "mlflow.tags.entropy_regularization_coef",
        )
        if col in summary_df.columns
    ),
    None,
)
if beta_col is None:
    raise ValueError("Cannot find entropy beta column in summary_df")

beta_by_run = pd.to_numeric(
    summary_df.set_index("run_name")[beta_col],
    errors="coerce",
)
history_df["entropy_beta"] = history_df["run_name"].map(beta_by_run)

lambda_values = pd.to_numeric(history_df["lambda_coef"], errors="coerce")

if "valid_negative_entropy" in history_df.columns:
    valid_negative_entropy = pd.to_numeric(
        history_df["valid_negative_entropy"],
        errors="coerce",
    )
    history_df["valid_entropy"] = -valid_negative_entropy
    history_df["valid_entropy_loss_term"] = (
        history_df["entropy_beta"] * valid_negative_entropy
    )

if "valid_mean_p_open" in history_df.columns:
    valid_mean_p_open = pd.to_numeric(
        history_df["valid_mean_p_open"],
        errors="coerce",
    )
    history_df["valid_lambda_p_open_loss"] = lambda_values * valid_mean_p_open
    clipped_p_open = valid_mean_p_open.clip(1e-8, 1.0 - 1e-8)
    history_df["valid_lambda_log_odds_loss_approx"] = lambda_values * np.log(
        (1.0 - clipped_p_open) / clipped_p_open
    )

if (
    "valid_lambda_p_open_loss" in history_df.columns
    and "valid_entropy_loss_term" in history_df.columns
):
    history_df["valid_estimated_gate_loss"] = (
        history_df["valid_lambda_p_open_loss"]
        + history_df["valid_entropy_loss_term"]
    )

In [ ]:
plot_metric(
    history_df,
    "lambda_coef",
    label_col="run_label",
    show_mean=True,
    title="lambda_coef / epoch by entropy beta",
    yscale="log",
)

In [ ]:
plot_metric(
    history_df,
    "valid_loss",
    label_col="run_label",
    show_mean=True,
    title="valid_loss / epoch by entropy beta",
)

In [ ]:
plot_metric(
    history_df,
    "valid_ce_loss",
    label_col="run_label",
    show_mean=True,
    title="valid_ce_loss / epoch by entropy beta",
)

In [ ]:
plot_metric(
    history_df,
    "valid_reg_loss",
    label_col="run_label",
    show_mean=True,
    title="valid_reg_loss / epoch by entropy beta",
)

In [ ]:
plot_metric(
    history_df,
    "valid_regularization_loss",
    label_col="run_label",
    show_mean=True,
    title="valid_regularization_loss / epoch by entropy beta",
)

In [ ]:
plot_metric(
    history_df,
    "valid_lambda_p_open_loss",
    label_col="run_label",
    show_mean=True,
    title="lambda * mean_p_open / epoch by entropy beta",
    yscale="log",
)

In [ ]:
plot_metric(
    history_df,
    "valid_lambda_log_odds_loss_approx",
    label_col="run_label",
    show_mean=True,
    title="lambda * log((1-p_open)/p_open) approx / epoch by entropy beta",
)

In [ ]:
plot_metric(
    history_df,
    "valid_entropy_loss_term",
    label_col="run_label",
    show_mean=True,
    title="beta * valid_negative_entropy / epoch by entropy beta",
)

In [ ]:
plot_metric(
    history_df,
    "valid_estimated_gate_loss",
    label_col="run_label",
    show_mean=True,
    title="estimated gate loss contribution / epoch by entropy beta",
)

In [ ]:
plot_metric(
    history_df,
    "valid_active_blocks_expected",
    label_col="run_label",
    show_mean=True,
    title="expected active blocks / epoch by entropy beta",
)

In [ ]:
plot_metric(
    history_df,
    "valid_mean_gate_prob",
    label_col="run_label",
    show_mean=True,
    title="valid_mean_gate_prob / epoch by entropy beta",
)

In [ ]:
plot_metric(
    history_df,
    "valid_mean_p_open",
    label_col="run_label",
    show_mean=True,
    title="valid_mean_p_open / epoch by entropy beta",
)

In [ ]:
plot_metric(
    history_df,
    "grad_norm_ce_total_mean",
    label_col="run_label",
    show_mean=True,
    title="grad_norm_ce_total_mean / epoch by entropy beta",
    yscale="log",
)

In [ ]:
plot_metric(
    history_df,
    "grad_norm_regularization_total_mean",
    label_col="run_label",
    show_mean=True,
    title="grad_norm_regularization_total_mean / epoch by entropy beta",
    yscale="log",
)

In [ ]:
plot_metric(
    history_df,
    "grad_norm_total_total_mean",
    label_col="run_label",
    show_mean=True,
    title="grad_norm_total_total_mean / epoch by entropy beta",
    yscale="log",
)

In [ ]:
plot_metric(
    history_df,
    "grad_norm_ce_gumbel_logits_total_mean",
    label_col="run_label",
    show_mean=True,
    title="grad_norm_ce_gumbel_logits_total_mean / epoch by entropy beta",
    yscale="log",
)

In [ ]:
plot_metric(
    history_df,
    "grad_norm_regularization_gumbel_logits_total_mean",
    label_col="run_label",
    show_mean=True,
    title="grad_norm_regularization_gumbel_logits_total_mean / epoch by entropy beta",
    yscale="log",
)